In [1]:
# ============================================================================
# AFDB ANNOTATION EXPLORATION NOTEBOOK
# Purpose: Understand the structure of MIT-BIH AFDB annotations
# Save as: notebooks/afdb_annotation_explorer.ipynb
# ============================================================================

# Cell 1: Imports
import sys
sys.path.append('../src')
from data_loader import UniversalECGLoader
import numpy as np
import pandas as pd
from pathlib import Path

print("✅ Imports complete")

✅ Imports complete


In [2]:
# ============================================================================
# Cell 2: Load AFDB Dataset
# ============================================================================

afdb_path = '../data/raw/mit-bih-afdb/'
afdb_loader = UniversalECGLoader(afdb_path)
afdb_records = afdb_loader.get_valid_records()

print(f"\n📊 Found {len(afdb_records)} valid AFDB records")
print(f"First 5 records: {afdb_records[:5]}")



ANALYZING DATASET AT: ..\data\raw\mit-bih-afdb

📁 Directory Structure:
   Root: ..\data\raw\mit-bih-afdb
   Subdirectories: old

📄 File Types Found:
   .atr: 50 files
   .atr-: 2 files
   .dat: 23 files
   .hea: 25 files
   .hea-: 23 files
   .qrs: 25 files
   .qrs-: 1 files
   .qrsc: 2 files
   .txt: 2 files
   .xws: 1 files

📊 Metadata Files:
   - notes.txt
   - SHA256SUMS.txt

🔍 Detected Format: WFDB
   Total record files: 23


✅ Found 25 records in WFDB format

🔍 Validating 25 records...
   Progress: 10/25 (8 valid)
   Progress: 20/25 (18 valid)

✅ Found 23 valid records
⚠️  Skipped 2 problematic records:
   • 00735: Failed to load record '00735': Error reading WFDB file: samp...
   • 03665: Failed to load record '03665': Error reading WFDB file: samp...

📊 Found 23 valid AFDB records
First 5 records: ['04015', '04043', '04048', '04126', '04746']


In [3]:
# ============================================================================
# Cell 3: Load and Inspect a Single Record
# ============================================================================

if afdb_records:
    test_record = afdb_records[0]
    print(f"\n{'='*70}")
    print(f"LOADING RECORD: {test_record}")
    print(f"{'='*70}\n")
    
    # Load the record
    record_data = afdb_loader.load_record(test_record)
    
    # Print all top-level keys
    print(f"📋 Top-level keys in record_data:")
    for key in record_data.keys():
        print(f"   • {key}")
    
    # Print basic info
    print(f"\n📊 Signal Information:")
    print(f"   Shape: {record_data['signal'].shape}")
    print(f"   Sampling rate: {record_data['fs']} Hz")
    print(f"   Duration: {record_data['duration']:.2f} seconds")
    print(f"   Channels: {record_data['channels']}")
    
    # Check if annotations exist
    if 'annotations' in record_data:
        print(f"\n✅ Annotations found!")
    else:
        print(f"\n❌ No annotations found!")
else:
    print("❌ No AFDB records available!")




LOADING RECORD: 04015

📋 Top-level keys in record_data:
   • signal
   • fs
   • channels
   • units
   • duration
   • format
   • annotations

📊 Signal Information:
   Shape: (9205760, 2)
   Sampling rate: 250 Hz
   Duration: 36823.04 seconds
   Channels: ['ECG1', 'ECG2']

✅ Annotations found!


In [4]:
# ============================================================================
# Cell 4: Deep Dive into Annotations Structure
# ============================================================================

if afdb_records and 'annotations' in record_data:
    annotations = record_data['annotations']
    
    print(f"\n{'='*70}")
    print(f"ANNOTATION STRUCTURE")
    print(f"{'='*70}\n")
    
    print(f"🔍 Annotation keys: {list(annotations.keys())}\n")
    
    # Inspect each key in annotations
    for key, value in annotations.items():
        print(f"\n📌 Key: '{key}'")
        print(f"   Type: {type(value)}")
        
        if isinstance(value, (list, np.ndarray)):
            print(f"   Length: {len(value)}")
            print(f"   First 10 values: {value[:10]}")
            
            # If it's aux_notes, show more details
            if 'aux' in key.lower() or 'note' in key.lower():
                print(f"\n   👉 This looks like rhythm annotations!")
                unique_values = set(str(v) for v in value if v)
                print(f"   Unique rhythm types found: {unique_values}")
        else:
            print(f"   Value: {value}")


ANNOTATION STRUCTURE

🔍 Annotation keys: ['samples', 'symbols', 'aux_notes']


📌 Key: 'samples'
   Type: <class 'numpy.ndarray'>
   Length: 15
   First 10 values: [     30  102584  119604  121773  122194  133348  166857 1096245 1098054
 1135296]

📌 Key: 'symbols'
   Type: <class 'list'>
   Length: 15
   First 10 values: ['+', '+', '+', '+', '+', '+', '+', '+', '+', '+']

📌 Key: 'aux_notes'
   Type: <class 'list'>
   Length: 15
   First 10 values: ['(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB']

   👉 This looks like rhythm annotations!
   Unique rhythm types found: {'(AFIB', '(N'}


In [5]:
# ============================================================================
# Cell 5: Create a Timeline of Rhythm Changes
# ============================================================================

if afdb_records and 'annotations' in record_data:
    annotations = record_data['annotations']
    
    # Try to find the correct keys
    sample_key = None
    aux_key = None
    
    for key in annotations.keys():
        if 'sample' in key.lower():
            sample_key = key
        if 'aux' in key.lower() or 'note' in key.lower():
            aux_key = key
    
    print(f"\n{'='*70}")
    print(f"RHYTHM TIMELINE")
    print(f"{'='*70}\n")
    
    if sample_key and aux_key:
        samples = annotations[sample_key]
        aux_notes = annotations[aux_key]
        
        print(f"✅ Found sample key: '{sample_key}'")
        print(f"✅ Found aux_notes key: '{aux_key}'")
        print(f"\nTotal rhythm changes: {len(samples)}\n")
        
        # Create a timeline
        print(f"{'Sample Index':<15} {'Time (min)':<12} {'Rhythm':<20}")
        print(f"{'-'*50}")
        
        for i in range(min(20, len(samples))):  # Show first 20 changes
            sample_idx = samples[i]
            time_seconds = sample_idx / record_data['fs']
            time_minutes = time_seconds / 60
            rhythm = str(aux_notes[i]) if i < len(aux_notes) else 'N/A'
            
            print(f"{sample_idx:<15} {time_minutes:<12.2f} {rhythm:<20}")
        
        if len(samples) > 20:
            print(f"\n... and {len(samples) - 20} more rhythm changes")
    else:
        print(f"❌ Could not find sample or aux_notes keys!")
        print(f"Available keys: {list(annotations.keys())}")




RHYTHM TIMELINE

✅ Found sample key: 'samples'
✅ Found aux_notes key: 'aux_notes'

Total rhythm changes: 15

Sample Index    Time (min)   Rhythm              
--------------------------------------------------
30              0.00         (N                  
102584          6.84         (AFIB               
119604          7.97         (N                  
121773          8.12         (AFIB               
122194          8.15         (N                  
133348          8.89         (AFIB               
166857          11.12        (N                  
1096245         73.08        (AFIB               
1098054         73.20        (N                  
1135296         75.69        (AFIB               
1139595         75.97        (N                  
1422436         94.83        (AFIB               
1423548         94.90        (N                  
1459277         97.29        (AFIB               
1460416         97.36        (N                  


In [6]:
# ============================================================================
# Cell 6: Analyze Rhythm Distribution
# ============================================================================

if afdb_records and 'annotations' in record_data:
    annotations = record_data['annotations']
    
    if sample_key and aux_key:
        samples = annotations[sample_key]
        aux_notes = annotations[aux_key]
        
        print(f"\n{'='*70}")
        print(f"RHYTHM DISTRIBUTION ANALYSIS")
        print(f"{'='*70}\n")
        
        # Count rhythm types
        rhythm_counts = {}
        rhythm_durations = {}
        
        for i in range(len(samples)):
            rhythm = str(aux_notes[i]) if i < len(aux_notes) else ''
            
            # Simplify rhythm label
            if '(AFIB' in rhythm or '(AFL' in rhythm:
                rhythm_simple = 'AFib/Flutter'
            elif '(N' in rhythm:
                rhythm_simple = 'Normal'
            else:
                rhythm_simple = rhythm if rhythm else 'Unknown'
            
            # Count occurrences
            rhythm_counts[rhythm_simple] = rhythm_counts.get(rhythm_simple, 0) + 1
            
            # Calculate duration (time until next change)
            if i < len(samples) - 1:
                duration_samples = samples[i + 1] - samples[i]
            else:
                # Last segment until end of recording
                duration_samples = len(record_data['signal']) - samples[i]
            
            duration_seconds = duration_samples / record_data['fs']
            rhythm_durations[rhythm_simple] = rhythm_durations.get(rhythm_simple, 0) + duration_seconds
        
        # Print counts
        print(f"📊 Rhythm Change Counts:")
        for rhythm, count in sorted(rhythm_counts.items()):
            print(f"   {rhythm:<20}: {count:>5} changes")
        
        # Print durations
        total_duration = sum(rhythm_durations.values())
        print(f"\n⏱️  Time Spent in Each Rhythm:")
        for rhythm, duration in sorted(rhythm_durations.items()):
            percentage = (duration / total_duration) * 100
            duration_min = duration / 60
            print(f"   {rhythm:<20}: {duration_min:>8.2f} min ({percentage:>5.1f}%)")
        
        print(f"\n   Total recording: {total_duration/60:.2f} minutes")



RHYTHM DISTRIBUTION ANALYSIS

📊 Rhythm Change Counts:
   AFib/Flutter        :     7 changes
   Normal              :     8 changes

⏱️  Time Spent in Each Rhythm:
   AFib/Flutter        :     3.95 min (  0.6%)
   Normal              :   609.76 min ( 99.4%)

   Total recording: 613.72 minutes


In [7]:
# ============================================================================
# Cell 7: Test Pure Window Labeling Logic
# ============================================================================

if afdb_records and 'annotations' in record_data and sample_key and aux_key:
    
    print(f"\n{'='*70}")
    print(f"TESTING PURE WINDOW LABELING LOGIC")
    print(f"{'='*70}\n")
    
    samples = annotations[sample_key]
    aux_notes = annotations[aux_key]
    
    def get_rhythm_at_sample(sample_idx):
        """Get rhythm at a specific sample index"""
        current_rhythm = 'N'
        
        for i, sample in enumerate(samples):
            if sample <= sample_idx:
                rhythm_str = str(aux_notes[i]) if i < len(aux_notes) else ''
                
                if '(AFIB' in rhythm_str or '(AFL' in rhythm_str:
                    current_rhythm = 'A'
                elif '(N' in rhythm_str or rhythm_str == '':
                    current_rhythm = 'N'
            else:
                break
        
        return current_rhythm
    
    def get_pure_window_label(start_sample, end_sample):
        """
        Label window ONLY if it contains a single rhythm throughout.
        Returns None if the window contains a rhythm transition.
        """
        # Step 1: Get rhythm at window start
        rhythm_at_start = get_rhythm_at_sample(start_sample)
        
        # Step 2: Check if ANY rhythm changes occur INSIDE the window
        for change_sample in samples:
            if start_sample < change_sample < end_sample:
                # Transition detected inside window!
                return None  # This is a MIXED window
        
        # Step 3: Window is pure - return the rhythm
        return rhythm_at_start
    
    # Test at different time points using WINDOWS instead of single points
    WINDOW_SECONDS = 10
    test_times = [0, 30, 60, 120, 300, 600, 1800, 3600]  # seconds
    
    print(f"Testing pure window labeling at different time points:\n")
    print(f"{'Time (min)':<12} {'Window Start':<15} {'Window End':<15} {'Label':<10} {'Status':<15}")
    print(f"{'-'*75}")
    
    for time_sec in test_times:
        start_sample = int(time_sec * record_data['fs'])
        end_sample = int((time_sec + WINDOW_SECONDS) * record_data['fs'])
        
        if end_sample < len(record_data['signal']):
            label = get_pure_window_label(start_sample, end_sample)
            time_min = time_sec / 60
            
            if label is not None:
                status = f"Pure {label}"
            else:
                status = "MIXED (discard)"
            
            print(f"{time_min:<12.2f} {start_sample:<15} {end_sample:<15} {str(label):<10} {status:<15}")


TESTING PURE WINDOW LABELING LOGIC

Testing pure window labeling at different time points:

Time (min)   Window Start    Window End      Label      Status         
---------------------------------------------------------------------------
0.00         0               2500            None       MIXED (discard)
0.50         7500            10000           N          Pure N         
1.00         15000           17500           N          Pure N         
2.00         30000           32500           N          Pure N         
5.00         75000           77500           N          Pure N         
10.00        150000          152500          A          Pure A         
30.00        450000          452500          N          Pure N         
60.00        900000          902500          N          Pure N         


In [8]:
# ============================================================================
# Cell 8: Extract Pure Window Labels for This Record
# ============================================================================

if afdb_records and 'annotations' in record_data and sample_key and aux_key:
    
    print(f"\n{'='*70}")
    print(f"EXTRACTING PURE WINDOW LABELS (10-second windows)")
    print(f"{'='*70}\n")
    
    WINDOW_SECONDS = 10
    TARGET_FS = 250  # After resampling
    SAMPLES_PER_WINDOW = WINDOW_SECONDS * TARGET_FS
    
    # Calculate total windows in this record
    signal_length_samples = len(record_data['signal'])
    # After resampling to 250 Hz
    resampled_length = int(signal_length_samples * (TARGET_FS / record_data['fs']))
    num_windows = resampled_length // SAMPLES_PER_WINDOW
    
    print(f"Original signal length: {signal_length_samples:,} samples")
    print(f"After resampling to {TARGET_FS} Hz: {resampled_length:,} samples")
    print(f"Total 10-second windows: {num_windows}")
    
    # Get label for each window
    window_labels = []
    discarded_windows = []  # Track which windows were discarded
    scale_factor = TARGET_FS / record_data['fs']
    
    for window_idx in range(num_windows):
        # Calculate window boundaries (in resampled signal)
        start_sample_resampled = window_idx * SAMPLES_PER_WINDOW
        end_sample_resampled = start_sample_resampled + SAMPLES_PER_WINDOW
        
        # Convert back to original sampling rate for annotation lookup
        start_sample_original = int(start_sample_resampled / scale_factor)
        end_sample_original = int(end_sample_resampled / scale_factor)
        
        # Get pure window label (or None if mixed)
        label = get_pure_window_label(start_sample_original, end_sample_original)
        
        if label is not None:
            window_labels.append(label)
        else:
            # Track discarded windows for analysis
            discarded_windows.append(window_idx)
    
    # Count labels
    label_counts = {'A': 0, 'N': 0}
    for label in window_labels:
        label_counts[label] = label_counts.get(label, 0) + 1
    
    discarded_count = len(discarded_windows)
    kept_count = len(window_labels)
    afib_count = label_counts.get('A', 0)
    normal_count = label_counts.get('N', 0)
    
    print(f"\n📊 Window Label Distribution:")
    print(f"   Pure AFib windows: {afib_count} ({afib_count/kept_count*100:.2f}% of kept windows)")
    print(f"   Pure Normal windows: {normal_count} ({normal_count/kept_count*100:.2f}% of kept windows)")
    print(f"   Mixed windows (discarded): {discarded_count}")
    
    print(f"\n📈 Window Statistics:")
    print(f"   Total windows: {num_windows}")
    print(f"   Kept (pure): {kept_count} ({kept_count/num_windows*100:.2f}%)")
    print(f"   Discarded (mixed): {discarded_count} ({discarded_count/num_windows*100:.2f}%)")
    
    print(f"\n✅ Successfully labeled {kept_count} pure windows!")
    print(f"   First 20 labels: {window_labels[:20]}")
    
    # Show details of discarded windows (optional - for debugging)
    if discarded_count > 0 and discarded_count <= 20:
        print(f"\n🔍 Discarded window indices: {discarded_windows}")
    elif discarded_count > 20:
        print(f"\n🔍 First 20 discarded window indices: {discarded_windows[:20]}")
        print(f"   ... and {discarded_count - 20} more")


EXTRACTING PURE WINDOW LABELS (10-second windows)

Original signal length: 9,205,760 samples
After resampling to 250 Hz: 9,205,760 samples
Total 10-second windows: 3682

📊 Window Label Distribution:
   Pure AFib windows: 17 (0.46% of kept windows)
   Pure Normal windows: 3651 (99.54% of kept windows)
   Mixed windows (discarded): 14

📈 Window Statistics:
   Total windows: 3682
   Kept (pure): 3668 (99.62%)
   Discarded (mixed): 14 (0.38%)

✅ Successfully labeled 3668 pure windows!
   First 20 labels: ['N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N']

🔍 Discarded window indices: [0, 41, 47, 48, 53, 66, 438, 439, 454, 455, 568, 569, 583, 584]


In [9]:
# ============================================================================
# Cell 9: Process All AFDB Records - Extract Window Labels
# ============================================================================

print(f"\n{'='*70}")
print(f"PROCESSING ALL AFDB RECORDS")
print(f"{'='*70}\n")

all_afdb_window_labels = []
afdb_summary = []

for record_id in afdb_records:
    try:
        # Load record
        record_data = afdb_loader.load_record(record_id)
        
        if 'annotations' not in record_data:
            print(f"⚠️  {record_id}: No annotations, skipping")
            continue
        
        annotations = record_data['annotations']
        
        # Find correct keys
        sample_key = None
        aux_key = None
        for key in annotations.keys():
            if 'sample' in key.lower():
                sample_key = key
            if 'aux' in key.lower() or 'note' in key.lower():
                aux_key = key
        
        if not sample_key or not aux_key:
            print(f"⚠️  {record_id}: Missing annotation keys, skipping")
            continue
        
        samples = annotations[sample_key]
        aux_notes = annotations[aux_key]
        
        # Calculate windows
        signal_length = len(record_data['signal'])
        scale_factor = TARGET_FS / record_data['fs']
        resampled_length = int(signal_length * scale_factor)
        num_windows = resampled_length // SAMPLES_PER_WINDOW
        
        # Get labels for each window
        window_labels = []
        for window_idx in range(num_windows):
            window_middle_resampled = (window_idx * SAMPLES_PER_WINDOW) + (SAMPLES_PER_WINDOW // 2)
            window_middle_original = int(window_middle_resampled / scale_factor)
            
            # Get rhythm
            current_rhythm = 'N'
            for i, sample in enumerate(samples):
                if sample <= window_middle_original:
                    rhythm_str = str(aux_notes[i]) if i < len(aux_notes) else ''
                    if '(AFIB' in rhythm_str or '(AFL' in rhythm_str:
                        current_rhythm = 'A'
                    elif '(N' in rhythm_str or rhythm_str == '':
                        current_rhythm = 'N'
                else:
                    break
            
            window_labels.append(current_rhythm)
        
        # Count labels
        afib_count = sum(1 for l in window_labels if l == 'A')
        normal_count = sum(1 for l in window_labels if l == 'N')
        
        afdb_summary.append({
            'record_id': record_id,
            'total_windows': num_windows,
            'afib_windows': afib_count,
            'normal_windows': normal_count,
            'afib_percentage': (afib_count / num_windows * 100) if num_windows > 0 else 0
        })
        
        print(f"✅ {record_id}: {num_windows} windows ({afib_count} AFib, {normal_count} Normal)")
        
    except Exception as e:
        print(f"❌ {record_id}: Error - {str(e)[:60]}")

# Create summary DataFrame
afdb_summary_df = pd.DataFrame(afdb_summary)

print(f"\n{'='*70}")
print(f"AFDB SUMMARY")
print(f"{'='*70}\n")
print(afdb_summary_df)

print(f"\n📊 Overall AFDB Statistics:")
print(f"   Total records: {len(afdb_summary_df)}")
print(f"   Total windows: {afdb_summary_df['total_windows'].sum()}")
print(f"   Total AFib windows: {afdb_summary_df['afib_windows'].sum()}")
print(f"   Total Normal windows: {afdb_summary_df['normal_windows'].sum()}")

if len(afdb_summary_df) > 0:
    total_windows = afdb_summary_df['total_windows'].sum()
    total_afib = afdb_summary_df['afib_windows'].sum()
    total_normal = afdb_summary_df['normal_windows'].sum()
    
    print(f"\n   Overall AFib percentage: {total_afib/total_windows*100:.1f}%")
    print(f"   Overall Normal percentage: {total_normal/total_windows*100:.1f}%")



PROCESSING ALL AFDB RECORDS

✅ 04015: 3682 windows (24 AFib, 3658 Normal)
✅ 04043: 3682 windows (797 AFib, 2885 Normal)
✅ 04048: 3682 windows (35 AFib, 3647 Normal)
✅ 04126: 3682 windows (138 AFib, 3544 Normal)
✅ 04746: 3682 windows (1954 AFib, 1728 Normal)
✅ 04908: 3682 windows (333 AFib, 3349 Normal)
✅ 04936: 3682 windows (2994 AFib, 688 Normal)
✅ 05091: 3682 windows (8 AFib, 3674 Normal)
✅ 05121: 3682 windows (2345 AFib, 1337 Normal)
✅ 05261: 3682 windows (48 AFib, 3634 Normal)
✅ 06426: 3682 windows (3533 AFib, 149 Normal)
✅ 06453: 3330 windows (37 AFib, 3293 Normal)
✅ 06995: 3682 windows (1737 AFib, 1945 Normal)
✅ 07162: 3682 windows (3682 AFib, 0 Normal)
✅ 07859: 3682 windows (3682 AFib, 0 Normal)
✅ 07879: 3682 windows (2221 AFib, 1461 Normal)
✅ 07910: 3682 windows (635 AFib, 3047 Normal)
✅ 08215: 3682 windows (2972 AFib, 710 Normal)
✅ 08219: 3682 windows (797 AFib, 2885 Normal)
✅ 08378: 3682 windows (925 AFib, 2757 Normal)
✅ 08405: 3682 windows (2659 AFib, 1023 Normal)
✅ 08434: 